In [1]:
# =============================================================================
# RandomForest -- Nutrient-Aware Pathogen Suppression Predictor 0101
# Version 1 (Full), Dataset v8
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupKFold
from sklearn.linear_model import Ridge, LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ── Section 2: load data ────────────────────────────────────────────────
df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\Updated01\community_index_v8_backup.csv")
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# ── Section 3: feature engineering (identical to XGBoost V1) ──────────
species_cols    = [f"Species_{i:04d}" for i in range(2, 102)]
r0_cols         = [f"R0_c{k}" for k in range(1, 21)]
active_cols     = [f"active_c{k}" for k in range(1, 21)]
structural_cols = ['community_size', 'diversity_level']
feature_cols    = species_cols + r0_cols + active_cols + structural_cols

X = df[feature_cols]
y = df['community_suppression']
print(f"X shape: {X.shape}    y shape: {y.shape}")

# ── Section 4: train/test split (identical to XGBoost V1) ─────────────
strat_key = df['community_size'].astype(str) + "_" + df['diversity_level'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)
groups = df['broth_id'].values  # used later for GroupKFold

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(f"Unique broths: {df['broth_id'].nunique()}")

Dataset shape: 30,000 rows x 168 columns
X shape: (30000, 142)    y shape: (30000,)
Train set: 24,000 rows
Test set:  6,000 rows
Unique broths: 20


In [3]:
# ── Section 6: RandomForest initial fit ─────────────────────────────────
# RandomForest builds many independent trees in parallel (not sequentially
# like boosting), each on a bootstrap sample of the data, then averages
# their predictions. There is no early stopping because trees do not
# correct each other's errors -- each tree is independent.
print("\nRandomForest (initial)")

model_init = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,        # let trees grow fully; RF relies on averaging, not pruning, to control variance
    min_samples_leaf=3,    # similar role to XGBoost's min_child_weight
    max_features=0.8,      # fraction of features considered per split, similar role to colsample_bytree
    bootstrap=True,
    oob_score=True,        # out-of-bag score: a free validation estimate using the ~37% of rows
                            # each tree does not see in its bootstrap sample
    random_state=42,
    n_jobs=-1,
    verbose=0
)
model_init.fit(X_train, y_train)

y_pred_init = model_init.predict(X_test)
r2_init     = r2_score(y_test, y_pred_init)
mae_init    = mean_absolute_error(y_test, y_pred_init)
mse_init    = mean_squared_error(y_test, y_pred_init)

print(f"RandomForest (initial):  R² = {r2_init:.4f}   MAE = {mae_init:.4f}   MSE = {mse_init:.5f}")
print(f"OOB score (R² on held-out-by-bootstrap samples): {model_init.oob_score_:.4f}")


RandomForest (initial)
RandomForest (initial):  R² = 0.9522   MAE = 0.0368   MSE = 0.00502
OOB score (R² on held-out-by-bootstrap samples): 0.9538


In [5]:
# ── Section 7: hyperparameter tuning ────────────────────────────────────
print("\nHyperparameter tuning")

param_grid = {
    'max_depth':        [None, 20, 30],
    'min_samples_leaf':  [1, 3, 5],
    'max_features':      [0.6, 0.8, 1.0],
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

base_estimator = RandomForestRegressor(
    n_estimators=200,   # kept low during search to save time, raised later
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

grid_search = GridSearchCV(
    estimator=base_estimator,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"\nBest parameters found:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")

best_cv_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best CV RMSE: {best_cv_rmse:.4f}")


Hyperparameter tuning
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Best parameters found:
  max_depth: None
  max_features: 1.0
  min_samples_leaf: 1
Best CV RMSE: 0.0744


In [7]:
# ── Section 8: final model ──────────────────────────────────────────────
best_params = grid_search.best_params_

model_final = RandomForestRegressor(
    **best_params,
    n_estimators=1000,
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
model_final.fit(X_train, y_train)

y_pred_final = model_final.predict(X_test)
r2_final     = r2_score(y_test, y_pred_final)
mae_final    = mean_absolute_error(y_test, y_pred_final)
mse_final    = mean_squared_error(y_test, y_pred_final)
rmse_final   = np.sqrt(mse_final)

print(f"\nFinal Model Performance")
print(f"R²:   {r2_final:.4f}")
print(f"MAE:  {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"MSE:  {mse_final:.5f}")
print(f"OOB R²: {model_final.oob_score_:.4f}")

print("\nModel Comparison")
print(f"{'Model':<30} {'R²':>8}  {'MAE':>8}")
print("-" * 48)

print(f"{'RandomForest (initial)':<30} {r2_init:>8.4f}  {mae_init:>8.4f}")
print(f"{'RandomForest (tuned)':<30} {r2_final:>8.4f}  {mae_final:>8.4f}")

# ── ADDITION: "training curve" equivalent for RandomForest ─────────────
# RandomForest has no per-round loss because trees are independent, not
# sequential. The closest equivalent is plotting error against the
# NUMBER OF TREES used, which shows how averaging more trees reduces
# variance. We refit with warm_start to add trees incrementally and
# record OOB error at each step. This is slower than reading an attribute
# but is the correct way to produce an analogous curve for RandomForest.
print("\nBuilding tree-count curve (RandomForest equivalent of training curve)")

tree_counts  = list(range(10, 1001, 20))
oob_errors   = []
test_errors  = []

curve_model = RandomForestRegressor(
    **best_params,
    n_estimators=10,
    bootstrap=True,
    oob_score=True,
    warm_start=True,   # allows adding more trees without refitting from scratch
    random_state=42,
    n_jobs=-1,
    verbose=0
)

for n_trees in tree_counts:
    curve_model.set_params(n_estimators=n_trees)
    curve_model.fit(X_train, y_train)
    oob_pred = curve_model.oob_prediction_
    oob_rmse = np.sqrt(mean_squared_error(y_train, oob_pred))
    test_pred = curve_model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    oob_errors.append(oob_rmse)
    test_errors.append(test_rmse)

fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(tree_counts, oob_errors,  color='steelblue',  lw=1.5, label='OOB RMSE (train-side estimate)')
ax_tc.plot(tree_counts, test_errors, color='darkorange', lw=1.5, label='Test RMSE')
ax_tc.set_xlabel('Number of trees')
ax_tc.set_ylabel('RMSE')
ax_tc.set_title(f'RandomForest Error vs Tree Count  --  final test RMSE={test_errors[-1]:.4f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rf_training_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: rf_training_curve.png")


Final Model Performance
R²:   0.9531
MAE:  0.0370
RMSE: 0.0701
MSE:  0.00492
OOB R²: 0.9544

Model Comparison
Model                                R²       MAE
------------------------------------------------
RandomForest (initial)           0.9522    0.0368
RandomForest (tuned)             0.9531    0.0370

Building tree-count curve (RandomForest equivalent of training curve)
Saved: rf_training_curve.png


In [8]:
# ── ADDITION: GroupKFold broth-held-out evaluation ──────────────────────
gkf          = GroupKFold(n_splits=5)
fold_r2, fold_mae, fold_rmse = [], [], []

print("\nGroupKFold evaluation (held-out broths):")
for fold_num, (tr_idx, te_idx) in enumerate(
        gkf.split(X, y, groups=groups), start=1):

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    held_broths = sorted(set(df.iloc[te_idx]['broth_id']))
    held_divs   = sorted(int(d) for d in df.iloc[te_idx]['diversity_level'].unique())
    print(f"\nFold {fold_num}: broths={held_broths}  diversity levels={held_divs}  n_test={len(X_te):,}")

    m_fold = RandomForestRegressor(
        **best_params,
        n_estimators=1000,
        bootstrap=True,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    m_fold.fit(X_tr, y_tr)

    yp_fold = m_fold.predict(X_te)
    r2_f    = r2_score(y_te, yp_fold)
    mae_f   = mean_absolute_error(y_te, yp_fold)
    rmse_f  = np.sqrt(mean_squared_error(y_te, yp_fold))

    fold_r2.append(r2_f); fold_mae.append(mae_f); fold_rmse.append(rmse_f)
    print(f"  R²={r2_f:.4f}  MAE={mae_f:.4f}  RMSE={rmse_f:.4f}")

print(f"\nGroupKFold summary across {gkf.n_splits} folds:")
print(f"  R²:   mean={np.mean(fold_r2):.4f}  std={np.std(fold_r2):.4f}")
print(f"  MAE:  mean={np.mean(fold_mae):.4f}  std={np.std(fold_mae):.4f}")
print(f"  RMSE: mean={np.mean(fold_rmse):.4f}  std={np.std(fold_rmse):.4f}")
print(f"\nCompare to stratified-split R²={r2_final:.4f}")
print(f"Drop: {r2_final - np.mean(fold_r2):.4f}  (how much was broth memorisation)")

# ── Section 9: per-condition breakdown (identical structure to XGBoost) ─
print("\nPerformance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

def condition_metrics(y_true_arr, y_pred_arr):
    y_t      = np.asarray(y_true_arr)
    y_p      = np.asarray(y_pred_arr)
    r2       = r2_score(y_t, y_p)
    mae      = mean_absolute_error(y_t, y_p)
    rmse     = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _    = pearsonr(y_t, y_p)
    sr, _    = spearmanr(y_t, y_p)
    bias     = float(np.mean(y_p - y_t))
    std_true = float(np.std(y_t))
    return {'n': len(y_t), 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'bias': bias, 'std_true': std_true}

hdr = f"{'':6} {'R²':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} {'Bias':>8} {'std(y)':>7} {'N':>6}"
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m    = condition_metrics(grp['y_true'], grp['y_pred'])
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp['y_true'], grp['y_pred'])
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}")


GroupKFold evaluation (held-out broths):

Fold 1: broths=['Broth_05', 'Broth_10', 'Broth_15', 'Broth_20']  diversity levels=[2, 4, 8, 16]  n_test=6,000
  R²=-0.2807  MAE=0.1996  RMSE=0.2546

Fold 2: broths=['Broth_04', 'Broth_09', 'Broth_14', 'Broth_19']  diversity levels=[1, 4, 8, 16]  n_test=6,000
  R²=0.2598  MAE=0.1673  RMSE=0.2267

Fold 3: broths=['Broth_03', 'Broth_08', 'Broth_13', 'Broth_18']  diversity levels=[1, 2, 8, 16]  n_test=6,000
  R²=-2.2687  MAE=0.4069  RMSE=0.5490

Fold 4: broths=['Broth_02', 'Broth_07', 'Broth_12', 'Broth_17']  diversity levels=[1, 2, 4, 16]  n_test=6,000
  R²=0.5512  MAE=0.1212  RMSE=0.1806

Fold 5: broths=['Broth_01', 'Broth_06', 'Broth_11', 'Broth_16']  diversity levels=[1, 2, 4, 8]  n_test=6,000
  R²=0.2291  MAE=0.1316  RMSE=0.2038

GroupKFold summary across 5 folds:
  R²:   mean=-0.3019  std=1.0192
  MAE:  mean=0.2053  std=0.1045
  RMSE: mean=0.2830  std=0.1353

Compare to stratified-split R²=0.9531
Drop: 1.2550  (how much was broth memorisatio

In [ ]:
# ── Section 10: built-in feature importance ─────────────────────────────
# RandomForest's feature_importances_ uses mean decrease in impurity,
# same interface as XGBoost's, so this section is unchanged structurally.
print("\nRandomForest built-in feature importance")

importances        = model_final.feature_importances_
importance_series  = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

print("\nTop 20 most important features:")
print(importance_series.head(20).round(5).to_string())

species_imp = importance_series[species_cols].sum()
r0_imp      = importance_series[r0_cols].sum()
active_imp  = importance_series[active_cols].sum()
struct_imp  = importance_series[structural_cols].sum()

print(f"\nImportance grouped by feature type (total):")
print(f"  Species membership:   {species_imp:.4f}  ({100*species_imp:.1f}%)")
print(f"  R0 concentrations:    {r0_imp:.4f}  ({100*r0_imp:.1f}%)")
print(f"  Active flags:         {active_imp:.4f}  ({100*active_imp:.1f}%)")
print(f"  Structural (size/div):{struct_imp:.4f}  ({100*struct_imp:.1f}%)")

In [ ]:
# ── Section 11: SHAP ─────────────────────────────────────────────────────
# TreeExplainer supports RandomForestRegressor directly, same as XGBoost.
print("\nSHAP analysis")

explainer       = shap.TreeExplainer(model_final)
sample_idx      = np.random.default_rng(42).choice(len(X_test), size=2000, replace=False)
X_sample        = X_test.iloc[sample_idx]
shap_values     = explainer.shap_values(X_sample)
mean_abs_shap   = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nTop 20 features by mean |SHAP| value:")
print(shap_importance.head(20).round(5).to_string())

sp_shap     = shap_importance[species_cols].sum()
r0_shap     = shap_importance[r0_cols].sum()
act_shap    = shap_importance[active_cols].sum()
struct_shap = shap_importance[structural_cols].sum()
total_shap  = sp_shap + r0_shap + act_shap + struct_shap

print(f"\nSHAP importance by feature group:")
print(f"  Species membership:  {sp_shap:.4f}  ({100*sp_shap/total_shap:.1f}%)")
print(f"  R0 concentrations:   {r0_shap:.4f}  ({100*r0_shap/total_shap:.1f}%)")
print(f"  Active flags:        {act_shap:.4f}  ({100*act_shap/total_shap:.1f}%)")
print(f"  Structural:          {struct_shap:.4f}  ({100*struct_shap/total_shap:.1f}%)")

In [ ]:
# ── Section 12: figures + expanded global metrics ───────────────────────
print("\nGenerating plots -> rf_suppression_results.png")

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle('RandomForest Nutrient-Aware Suppression Predictor -- MiCRM v8',
             fontsize=15, fontweight='bold', y=0.98)

ax1 = fig.add_subplot(2, 3, 1)
ax1.scatter(y_test, y_pred_final, alpha=0.15, s=3, c='steelblue')
ax1.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Perfect prediction')
ax1.set_xlabel('True community suppression')
ax1.set_ylabel('Predicted community suppression')
ax1.set_title(f'Predicted vs Actual  (R²={r2_final:.3f})')
ax1.legend(fontsize=8)
ax1.set_xlim(-0.05, 1.05); ax1.set_ylim(-0.05, 1.05)

residuals = y_test.values - y_pred_final
ax2 = fig.add_subplot(2, 3, 2)
ax2.scatter(y_pred_final, residuals, alpha=0.15, s=3, c='darkorange')
ax2.axhline(0, color='r', linestyle='--', linewidth=1.5)
p5, p95 = np.percentile(residuals, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', linewidth=1, label=f'5th pct: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', linewidth=1, label=f'95th pct: {p95:.3f}')
ax2.set_xlabel('Predicted suppression')
ax2.set_ylabel('Residual (true - predicted)')
ax2.set_title('Residual Plot')
ax2.legend(fontsize=7)

ax3 = fig.add_subplot(2, 3, 3)
top20_shap = shap_importance.head(20)
colours = []
for feat in top20_shap.index:
    if feat in species_cols:  colours.append('#2196F3')
    elif feat in r0_cols:     colours.append('#4CAF50')
    elif feat in active_cols: colours.append('#FF9800')
    else:                     colours.append('#9C27B0')
ax3.barh(range(20), top20_shap.values[::-1], color=colours[::-1])
ax3.set_yticks(range(20))
ax3.set_yticklabels(top20_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title('Top 20 Features (SHAP)')
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='Species'),
    Patch(facecolor='#4CAF50', label='R0'),
    Patch(facecolor='#FF9800', label='Active'),
    Patch(facecolor='#9C27B0', label='Structural'),
]
ax3.legend(handles=legend_elements, loc='lower right', fontsize=7)

ax4 = fig.add_subplot(2, 3, 4)
div_levels = sorted(results_df['diversity_level'].unique())
r2_by_div = [r2_score(
    results_df[results_df['diversity_level']==d]['y_true'],
    results_df[results_df['diversity_level']==d]['y_pred']
) for d in div_levels]
ax4.bar([str(d) for d in div_levels], r2_by_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', linewidth=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level (# active nutrients)')
ax4.set_ylabel('R²')
ax4.set_title('Model R² by Diversity Level')
ax4.set_ylim(0, 1.05); ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
sizes = sorted(results_df['community_size'].unique())
r2_by_size = [r2_score(
    results_df[results_df['community_size']==n]['y_true'],
    results_df[results_df['community_size']==n]['y_pred']
) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_by_size, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', linewidth=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N')
ax5.set_ylabel('R²')
ax5.set_title('Model R² by Community Size')
ax5.set_ylim(0, 1.05); ax5.legend(fontsize=8)

ax6 = fig.add_subplot(2, 3, 6)
group_names   = ['Species\nmembership', 'R0\nconcentrations', 'Active\nflags', 'Structural']
group_shap    = [sp_shap, r0_shap, act_shap, struct_shap]
group_colours = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
ax6.pie(group_shap, labels=group_names, colors=group_colours,
        autopct='%.1f%%', startangle=90, textprops={'fontsize': 9})
ax6.set_title('SHAP Importance by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('rf_suppression_results.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: rf_suppression_results.png")

def full_model_metrics(y_true, y_pred, label):
    y_t   = np.asarray(y_true)
    y_p   = np.asarray(y_pred)
    resid = np.abs(y_t - y_p)
    r2    = r2_score(y_t, y_p)
    mae   = mean_absolute_error(y_t, y_p)
    rmse  = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _ = pearsonr(y_t, y_p)
    sr, _ = spearmanr(y_t, y_p)
    return {'label': label, 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'median_ae': float(np.median(resid)), 'max_ae': float(np.max(resid)),
            'within_05': float(np.mean(resid <= 0.05)), 'within_10': float(np.mean(resid <= 0.10))}

metrics_rows = [
    full_model_metrics(y_test, y_pred_mean,  'Global mean (floor)'),
    full_model_metrics(y_test, y_pred_ridge, 'Ridge'),
    full_model_metrics(y_test, y_pred_lasso, 'Lasso'),
    full_model_metrics(y_test, y_pred_enet,  'Elastic Net'),
    full_model_metrics(y_test, y_pred_init,  'RandomForest initial'),
    full_model_metrics(y_test, y_pred_final, 'RandomForest tuned'),
]

print("\nExpanded global metrics")
hdr = (f"{'Model':<22} {'R2':>7} {'MAE':>7} {'RMSE':>7} "
       f"{'Pearson':>8} {'Spearman':>9} {'MedAE':>7} "
       f"{'MaxAE':>7} {'W.05':>6} {'W.10':>6}")
print(hdr)
print("-" * len(hdr))
for m in metrics_rows:
    print(f"{m['label']:<22} {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['median_ae']:>7.4f} {m['max_ae']:>7.4f} "
          f"{m['within_05']:>6.3f} {m['within_10']:>6.3f}")

In [ ]:
# ── Section 13: save model and predictions ──────────────────────────────
import joblib

joblib.dump(model_final, 'rf_model_v8.pkl')
print("\nModel saved: rf_model_v8.pkl")

pred_df = pd.DataFrame({
    'community_id':    df.loc[X_test.index, 'community_id'].values,
    'broth_id':        df.loc[X_test.index, 'broth_id'].values,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'residual':        y_test.values - y_pred_final,
})
pred_df.to_csv('rf_predictions_v8.csv', index=False)
print("Predictions saved: rf_predictions_v8.csv")

shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df.to_csv('rf_shap_values_v8.csv', index=False)
print("SHAP values saved: rf_shap_values_v8.csv")

In [ ]:
# ── Section 14: summary ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"Model:    RandomForestRegressor")
print(f"Dataset:  30,000 communities x 142 features (v8, Version 1)")
print(f"R²    = {r2_final:.4f}")
print(f"MAE   = {mae_final:.4f}")
print(f"RMSE  = {rmse_final:.4f}")
print(f"GroupKFold R² (held-out broths): mean={np.mean(fold_r2):.4f} std={np.std(fold_r2):.4f}")
print("=" * 60)